# GradCheck - Tree Of Shapes CFP

This notebook mirrors the CFP gradcheck notebooks for the tree-of-shapes backend. It keeps the image small so the checks are fast and deterministic.

In [1]:
from mtlearn import morphology
import mtlearn

import numpy as np
import torch
from torch.autograd import gradcheck

from mtlearn.layers import (
    ConnectedFilterPreprocessingImplicitJacobianFunction,
    ConnectedFilterPreprocessingLayer,
    ConnectedFilterPreprocessingLayerWithCPUTreeTraversal,
)

In [9]:
torch.set_printoptions(precision=10, sci_mode=False)

dtype = torch.float64
device = torch.device("cpu")
beta_f = 1.0
clamp_logits = False

device = "cuda" if torch.cuda.is_available() else "cpu"

device

'cpu'

## 1 - Small Tree-Of-Shapes Case

In [10]:
image =  np.array([ 
        [2, 2, 2, 2, 2, 2, 0, 0, 0, 0, 0, 0],
        [2, 5, 5, 5, 3, 3, 3, 1, 1, 1, 3, 0],
        [2, 5, 6, 5, 3, 3, 3, 1, 5, 1, 3, 0],
        [2, 5, 6, 5, 3, 3, 3, 1, 6, 1, 3, 0],
        [2, 5, 6, 5, 3, 3, 3, 1, 5, 1, 3, 0],
        [2, 5, 5, 5, 3, 3, 3, 1, 1, 1, 3, 0],
        [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 0],
        [2, 2, 2, 2, 2, 2, 3, 3, 3, 3, 3, 0],
        [6, 6, 6, 6, 6, 2, 4, 4, 4, 4, 4, 0],
        [6, 4, 6, 4, 6, 2, 4, 7, 4, 7, 4, 0],
        [6, 6, 6, 6, 6, 2, 4, 4, 4, 4, 4, 0],
        [2, 2, 2, 2, 2, 2, 0, 0, 0, 0, 0, 0]
    ],dtype=np.uint8)


attribute_spec = [morphology.AttributeType.AREA, morphology.AttributeType.COMPACTNESS]
interpolations = ["self-dual", "min4c-max8c", "min8c-max4c"]

In [11]:
def make_case(tos_interpolation):
    tree = morphology.build_tree(
        image,
        "tree-of-shapes",
        tos_interpolation=tos_interpolation,
    )

    attributes = morphology.compute_attributes(tree, attribute_spec)[1]
    attributes = torch.as_tensor(attributes, dtype=dtype, device=device)
    attributes = (attributes - attributes.mean(dim=0, keepdim=True)) / attributes.std(
        dim=0,
        unbiased=False,
        keepdim=True,
    ).clamp_min(1e-8)

    residues, tpre, tpost, parent, node_of_pixel = (
        mtlearn.ConnectedFilterPreprocessingTreeTensors.get_info_for_jacobian(tree)
    )

    return {
        "tree": tree,
        "attributes": attributes,
        "residues": residues.to(device=device, dtype=dtype),
        "tpre": tpre.to(device=device),
        "tpost": tpost.to(device=device),
        "parent": parent.to(device=device),
        "node_of_pixel": node_of_pixel.to(device=device),
    }

## 2 - Implicit-Jacobian Gradcheck

In [12]:
def filtered_mean(case, weight, bias):
    tree = case["tree"]
    return ConnectedFilterPreprocessingImplicitJacobianFunction.apply(
        weight,
        bias,
        case["residues"],
        case["tpre"],
        case["tpost"],
        case["parent"],
        case["node_of_pixel"],
        case["attributes"],
        tree.numRows,
        tree.numCols,
        beta_f,
        clamp_logits,
    ).mean()

In [13]:
gradcheck_results = {}

for interpolation in interpolations:
    case = make_case(interpolation)
    weight = torch.tensor([0.35, -0.2], dtype=dtype, device=device, requires_grad=True)
    bias = torch.tensor([0.1], dtype=dtype, device=device, requires_grad=True)

    gradcheck_results[interpolation] = gradcheck(
        lambda w, b: filtered_mean(case, w, b),
        (weight, bias),
        eps=1e-6,
        atol=1e-4,
    )

gradcheck_results

{'self-dual': True, 'min4c-max8c': True, 'min8c-max4c': True}

In [14]:
assert all(gradcheck_results.values())

## 3 - Layer-Level Consistency

In [15]:
x = torch.as_tensor(image, dtype=torch.float32).reshape(1, 1, *image.shape)

implicit_layer = ConnectedFilterPreprocessingLayer(
    in_channels=1,
    attributes_spec=[(morphology.AttributeType.AREA,)],
    tree_type="tree-of-shapes",
    tos_interpolation="min4c-max8c",
    device="cpu",
    scale_mode="none",
    beta_f=1.0,
    clamp_logits=False,
)

cpu_tree_layer = ConnectedFilterPreprocessingLayerWithCPUTreeTraversal(
    in_channels=1,
    attributes_spec=[(morphology.AttributeType.AREA,)],
    tree_type="tos",
    tos_interpolation="min4c-max8c",
    device="cpu",
    scale_mode="none",
    beta_f=1.0,
    clamp_logits=False,
)

with torch.no_grad():
    implicit_layer._weights["AREA"].fill_(0.2)
    implicit_layer._biases["AREA"].fill_(-0.1)
    cpu_tree_layer._weights["AREA"].copy_(implicit_layer._weights["AREA"])
    cpu_tree_layer._biases["AREA"].copy_(implicit_layer._biases["AREA"])

y_implicit = implicit_layer(x)
y_cpu_tree = cpu_tree_layer(x)

torch.allclose(y_implicit, y_cpu_tree), y_implicit, y_cpu_tree

(True,
 tensor([[[[2.0000000000, 2.0000000000, 2.0000000000, 2.0000000000, 2.0000000000,
            2.0000000000, 0.0140619874, 0.0140619874, 0.0140619874, 0.0140619874,
            0.0140619874, 0.0140619874],
           [2.0000000000, 4.8956909180, 4.8956909180, 4.8956909180, 2.9999980927,
            2.9999980927, 2.9999980927, 1.0006750822, 1.0006750822, 1.0006750822,
            2.9999980927, 0.0140619874],
           [2.0000000000, 4.8956909180, 5.5181503296, 4.8956909180, 2.9999980927,
            2.9999980927, 2.9999980927, 1.0006750822, 3.4905123711, 1.0006750822,
            2.9999980927, 0.0140619874],
           [2.0000000000, 4.8956909180, 5.5181503296, 4.8956909180, 2.9999980927,
            2.9999980927, 2.9999980927, 1.0006750822, 4.0154914856, 1.0006750822,
            2.9999980927, 0.0140619874],
           [2.0000000000, 4.8956909180, 5.5181503296, 4.8956909180, 2.9999980927,
            2.9999980927, 2.9999980927, 1.0006750822, 3.4905123711, 1.0006750822,
         

In [16]:
assert torch.allclose(y_implicit, y_cpu_tree)